# Data Cleaning & Data Manipulation

## Learning goals

By the end of this lesson, you should be able to:

- identify common data-quality problems;
- clean and standardize column names;
- standardize inconsistent categorical values;
- clean and transform string data;
- convert columns to appropriate data types;
- identify and handle missing values;
- identify and remove duplicate rows;
- verify that a cleaning step produced the expected result;
- use `map()` and `apply()` for a simple Series transformation;
- export cleaned data to a CSV file.

Throughout the lesson, we will use a simple workflow:

**Inspect → Decide → Clean → Verify**

## 1. Why do we clean data?

Real-world datasets are rarely ready for analysis.

Common problems include:

- inconsistent column names;
- inconsistent category labels;
- numbers stored as strings;
- unwanted characters;
- missing values;
- duplicated rows.

Cleaning data is not just about applying Pandas methods.

Before changing anything, ask:

> **What is wrong with this data, and why does it matter for the analysis?**

We will use an e-commerce dataset for the examples.

In [ ]:
import pandas as pd

orders = pd.DataFrame({
    "Order ID": [1001, 1002, 1003, 1003, 1004, 1005],
    "Customer City": ["madrid", "Madrid ", "BARCELONA", "BARCELONA", "Valencia", None],
    "Product Category": ["electronics", "Electronics", "home", "home", "Home", "ELECTRONICS"],
    "Price (€)": ["120.50", "89.99", "35.00", "35.00", "49.95", "199.00"],
    "Discount": ["10%", None, "5%", "5%", "0%", "15%"],
    "Units": [1, 2, 3, 3, None, 1]
})

display(orders)

## 2. Inspect before changing

Before cleaning, inspect the DataFrame.

Useful questions include:

- What are the column names?
- What data types does Pandas see?
- Are there missing values?
- Are there duplicates?
- Which categorical values look inconsistent?

### A useful diagnostic: unique values and frequencies

Before changing a categorical column, answer three different questions:

- **`unique()`** — *Which* distinct values are present? (Returns the values themselves.)
- **`nunique()`** — *How many* distinct values are present? (Returns a number; missing values are excluded by default.)
- **`value_counts(dropna=False)`** — *How often* does each value occur, including missing values?

These methods complement one another: they help us spot inconsistent labels and assess how widespread a problem is.

In [ ]:
display(orders["Customer City"].unique())
print("Number of distinct non-missing cities:", orders["Customer City"].nunique())
display(orders["Customer City"].value_counts(dropna=False))

In [ ]:
print("Columns:")
print(orders.columns.tolist())

print("\nData types:")
display(orders.dtypes)

print("\nMissing values:")
display(orders.isna().sum())

print("\nDuplicated rows:")
print(orders.duplicated().sum())

### 💡 Quick Check 1 — Explore another categorical column

The previous example examined **Customer City**. Now investigate **Product Category**, without cleaning it yet.

1. Which category labels appear?
2. How many distinct labels does Pandas currently recognize?
3. How frequently does each label occur?

**Think:** Are all the labels genuinely different categories, or could some be inconsistent spellings?

In [ ]:
# Distinct category labels
orders["Product Category"]._____()

# Number of distinct category labels
orders["Product Category"]._____()

# Frequency of each label
orders["Product Category"]._____(dropna=False)

## 3. Cleaning column names

Column names should be:

- consistent;
- easy to type;
- descriptive;
- free of unnecessary spaces.

A common convention is:

- lowercase;
- spaces replaced with `_`.

For example:

`"Customer City"` → `"customer_city"`

In [ ]:
orders.columns = (
    orders.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(orders.columns.tolist())

Sometimes a column name also needs a more descriptive replacement.

For individual changes, `rename()` is useful.

In [ ]:
orders = orders.rename(columns={
    "price_(€)": "price_eur"
})

print(orders.columns.tolist())

### 💡 Quick Check 2 — Complete the code

Clean these column names so that they are lowercase and spaces become underscores.

In [ ]:
example_columns = pd.DataFrame({
    "Customer Name": ["Ana"],
    "Order Date": ["2026-08-20"],
    "Total Price": [50]
})

example_columns.columns = (
    example_columns.columns
    .str.________()
    .str.________(" ", "_")
)

print(example_columns.columns.tolist())

## 4. Standardizing inconsistent categorical values

The same category may appear written in different ways:

- `"madrid"`
- `"Madrid "`
- `"MADRID"`

Pandas will treat them as different values.

Before cleaning a categorical column, inspect its unique values.

In [ ]:
print("Customer city values:")
print(orders["customer_city"].unique())

print("\nProduct category values:")
print(orders["product_category"].unique())

### String methods with `.str`

Pandas provides string methods through the `.str` accessor.

Useful methods include:

- `.str.strip()` → remove leading/trailing spaces;
- `.str.lower()` → lowercase;
- `.str.upper()` → uppercase;
- `.str.title()` → title case;
- `.str.replace()` → replace text.

In [ ]:
orders["customer_city"] = (
    orders["customer_city"]
    .str.strip()
    .str.title()
)

orders["product_category"] = (
    orders["product_category"]
    .str.strip()
    .str.title()
)

print(orders["customer_city"].unique())
print(orders["product_category"].unique())

### Mapping categories

Sometimes values are not just formatted differently: they need to be mapped to a common category.

### `map()` and `apply()` — the same transformation, two approaches

Last week we worked with Python functions. We can now reuse them with a Pandas Series.

- `Series.map()` can use a **dictionary** or a **function** to transform values one by one.
- `Series.apply()` can apply a **function** to each value of a Series.

For a simple element-wise function, both can produce the same result. We will use **the exact same function** to compare them.

In [ ]:
shipping_codes = pd.Series(["STD", "EXP", "STD", "EXP"])

def describe_shipping(code):
    return "Express" if code == "EXP" else "Standard"

mapped = shipping_codes.map(describe_shipping)
applied = shipping_codes.apply(describe_shipping)

comparison = pd.DataFrame({
    "original": shipping_codes,
    "with_map": mapped,
    "with_apply": applied
})

display(comparison)
print("Same results:", mapped.equals(applied))

**Which one should we choose?** For straightforward value-by-value transformations, `map()` is concise. `apply()` is also useful when we want to reuse a custom function. A dictionary-based `map()` is convenient for translating categories, but **unmapped values become missing (`NaN`)**. Use `replace()` if unmatched values should remain unchanged.

In [ ]:
shipping_with_unknown = pd.Series(["STD", "EXP", "ECO"])
shipping_dictionary = {"STD": "Standard", "EXP": "Express"}

print("map with dictionary:")
display(shipping_with_unknown.map(shipping_dictionary))

print("replace with dictionary:")
display(shipping_with_unknown.replace(shipping_dictionary))

In [ ]:
shipping = pd.Series(["STD", "EXP", "Standard", "Express"])

shipping_map = {
    "STD": "Standard",
    "EXP": "Express",
    "Standard": "Standard",
    "Express": "Express"
}

clean_shipping = shipping.map(shipping_map)

display(clean_shipping)

### 💡 Quick Check 3 — Complete the code

Standardize the following payment methods.

In [ ]:
payments = pd.Series(["CC", "Card", "PAYPAL", "Paypal"])

payment_map = {
    "CC": "Card",
    "Card": "Card",
    "PAYPAL": "Paypal",
    "Paypal": "Paypal"
}

clean_payments = payments.________(payment_map)

display(clean_payments)

## 5. Cleaning string values

Sometimes a string contains characters that should not be part of the final value.

For example, the `discount` column contains percentages as strings.

If we want to calculate with those values, the `%` symbol must be removed first.

In [ ]:
orders["discount_clean"] = (
    orders["discount"]
    .str.replace("%", "", regex=False)
)

display(orders[["discount", "discount_clean"]])

### Splitting strings

`str.split()` can separate pieces of a string using a delimiter.

In [ ]:
tracking_codes = pd.Series(["ES-MAD-001", "FR-PAR-002", "DE-BER-003"])

country_codes = tracking_codes.str.split("-").str[0]

display(country_codes)

### 💡 Quick Check 4 — Complete the code

Extract the middle part of each warehouse code.

In [ ]:
warehouse_codes = pd.Series(["MAD-NORTH-01", "BCN-SOUTH-02", "VAL-EAST-03"])

warehouse_area = warehouse_codes.str.________("-").str[_____]

display(warehouse_area)

## 6. Correcting data types

A column can look numeric while Pandas stores it as text.

Always check the current type before performing calculations.

In [ ]:
display(orders[["price_eur", "discount_clean"]].dtypes)

### `astype()`

If the values are already clean, `astype()` can convert the column.

### When values are not valid numbers: `pd.to_numeric()`

`astype(float)` works when all non-missing values can be converted. If a column contains invalid entries, it raises an error. `pd.to_numeric(..., errors="coerce")` converts invalid values to `NaN` so that we can inspect them and decide what to do next. **Coercion does not fix the original data.**

In [ ]:
raw_prices = pd.Series(["12.50", "8.00", "unknown", "19.95"])

numeric_prices = pd.to_numeric(raw_prices, errors="coerce")

display(pd.DataFrame({"original": raw_prices, "numeric": numeric_prices}))
print("Values requiring attention:", numeric_prices.isna().sum())

In [ ]:
orders["price_eur"] = orders["price_eur"].astype(float)

After changing a type, verify the result.

In [ ]:
print(orders["price_eur"].dtype)
print(orders["price_eur"].mean())

### 💡 Quick Check 5 — Complete the code

Convert the following string values to integers.

In [ ]:
units_text = pd.Series(["2", "5", "3", "8"])

units_numeric = units_text.________(_____)

print(units_numeric.dtype)

## 7. Missing values

Missing values represent information that is absent.

In Pandas they often appear as `NaN` or `None`.

The first step is not to delete them automatically.

The first step is to **inspect them**.

In [ ]:
display(orders.isna().sum())

### Choosing a strategy

There is no universal rule for missing values.

Possible strategies include:

- drop rows;
- drop columns;
- fill with a specific value;
- fill a numerical variable with mean or median;
- fill a categorical variable with its mode.

The correct choice depends on what the column means and how it will be used.

### Inspect missingness before choosing

A missing value is not necessarily zero. For example, an unknown discount may mean **no discount** in some systems, but in others it may mean **the discount was not recorded**. We must know what the field represents before filling it.

A practical first check is the proportion of missing values per column.

In [ ]:
missing_fraction = orders.isna().mean()
display(missing_fraction.sort_values(ascending=False))

### `dropna()`

Use `dropna()` when rows with missing information should not be kept.

For example, suppose a missing city makes an order unusable for a geographical analysis:

In [ ]:
orders_without_missing_city = orders.dropna(subset=["customer_city"])

display(orders_without_missing_city)

### `fillna()`

Sometimes we prefer to replace missing values.

For example, a missing discount can reasonably mean that no discount was applied:

In [ ]:
orders["discount_clean"] = orders["discount_clean"].fillna("0")
orders["discount_clean"] = orders["discount_clean"].astype(float)

display(orders[["discount", "discount_clean"]])

For numerical variables, mean or median may **sometimes** be appropriate.

For categorical variables, the mode is **sometimes** appropriate.

These choices should be justified rather than applied automatically.

In [ ]:
median_units = orders["units"].median()
orders["units_filled"] = orders["units"].fillna(median_units)

display(orders[["units", "units_filled"]])

### Verify after cleaning

Always check that the operation had the intended effect.

In [ ]:
display(orders[["discount_clean", "units_filled"]].isna().sum())

### 💡 Quick Check 6 — Complete the code

Fill the missing values in `stock` with the median, then verify that no missing values remain.

In [ ]:
inventory = pd.DataFrame({
    "product": ["A", "B", "C", "D"],
    "stock": [10, None, 6, 14]
})

median_stock = inventory["stock"].________()

inventory["stock"] = inventory["stock"].________(median_stock)

inventory["stock"].________().________()

## 8. Duplicates

Duplicated rows can accidentally count the same observation more than once.

Use `duplicated()` to identify them.

**Important distinction:** a completely duplicated row and a repeated business identifier are not necessarily the same problem. For example, several rows might legitimately share a customer ID because that customer placed multiple orders. Decide what uniquely identifies a record before removing anything.

In [ ]:
duplicate_mask = orders.duplicated()

display(duplicate_mask)
print("Number of duplicated rows:", duplicate_mask.sum())

### Removing duplicates

`drop_duplicates()` removes duplicated rows.

By default, Pandas keeps the first occurrence.

In [ ]:
orders_no_duplicates = orders.drop_duplicates()

print("Rows before:", len(orders))
print("Rows after:", len(orders_no_duplicates))

Duplicates can also be evaluated using specific columns with the `subset` parameter.

For example, **if** each order ID is meant to identify one unique record, you might check duplicates by ID. Inspect the records first: retaining the first occurrence is not always the correct business decision.

In [ ]:
orders_by_id = orders.drop_duplicates(
    subset=["order_id"],
    keep="first"
)

display(orders_by_id)

After dropping rows, the index may contain gaps.

`reset_index(drop=True)` creates a new sequential index.

In [ ]:
orders_by_id = orders_by_id.reset_index(drop=True)

display(orders_by_id)

### 💡 Quick Check 7 — Complete the code

Remove duplicated rows from the DataFrame and reset its index.

In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 2, 3],
    "city": ["Madrid", "Paris", "Paris", "Berlin"]
})

customers = customers.________()
customers = customers.________(drop=True)

display(customers)

## 9. Dropping unnecessary columns

Sometimes a column is not needed for the current analysis.

Use `drop(columns=...)` to remove it.

Do not remove columns simply because they look inconvenient. First decide whether they are relevant to the analysis.

In [ ]:
orders_reduced = orders.drop(columns=["discount"])

display(orders_reduced.head())

### 💡 Quick Check 8 — Complete the code

Remove the `temporary_note` column.

In [ ]:
products = pd.DataFrame({
    "product": ["A", "B", "C"],
    "price": [10, 20, 30],
    "temporary_note": ["check", "check", "check"]
})

products = products.________(columns=[________])

display(products)

## 10. The cleaning workflow

A reliable cleaning process follows a cycle:

### 1. Inspect

Identify the problem.

### 2. Decide

Ask what the correct representation should be.

### 3. Clean

Apply the appropriate transformation.

### 4. Verify

Check that the result is what you expected.

Cleaning without verification can silently introduce new problems.

### Saving a cleaned dataset

Once the data has been cleaned and verified, we can export it to a **CSV file**. This is useful when we want to reuse the cleaned dataset later without repeating every cleaning step.

The `to_csv()` method has several useful options:

- **`index=False`** — do not save the DataFrame's row index as an extra CSV column.
- **`sep=";"`** — separate fields with semicolons instead of commas. This can be useful for files exchanged with spreadsheet software or systems that expect this delimiter.
- **`encoding="utf-8"`** — write the file using UTF-8 character encoding, which supports accents and many other characters.

**Important:** When reading a CSV with a non-default separator, pass the same separator to `pd.read_csv()`. UTF-8 is the usual default, but specifying it explicitly makes the intended encoding clear.

In [ ]:
# Create a copy so the original example remains available.
cleaned_orders = orders_by_id.copy()

# Export with an explicit separator and encoding.
cleaned_orders.to_csv(
    "cleaned_orders.csv",
    index=False,
    sep=";",
    encoding="utf-8"
)

# Read it back using matching options.
reloaded_orders = pd.read_csv(
    "cleaned_orders.csv",
    sep=";",
    encoding="utf-8"
)

print("Exported shape:", reloaded_orders.shape)
display(reloaded_orders.head())

## Optional practice — Final Challenge — Complete the cleaning steps

A small product catalog has several data-quality problems.

Complete the missing code.

In [ ]:
catalog = pd.DataFrame({
    "Product Name": ["Keyboard", "Mouse", "Mouse", "Monitor"],
    "Category": ["electronics", "Electronics ", "Electronics ", "OFFICE"],
    "Price": ["49.99", "25.50", "25.50", "189.00"],
    "Stock": [10, None, None, 4]
})

# 1. Clean column names
catalog.columns = (
    catalog.columns
    .str.________()
    .str.________(" ", "_")
)

# 2. Standardize categories
catalog["category"] = (
    catalog["category"]
    .str.________()
    .str.________()
)

# 3. Convert price to numeric
catalog["price"] = catalog["price"].________(_____)

# 4. Fill missing stock values with the median
median_stock = catalog["stock"].________()
catalog["stock"] = catalog["stock"].________(median_stock)

# 5. Remove duplicated rows
catalog = catalog.________()

# 6. Reset the index
catalog = catalog.________(drop=True)

display(catalog)

## Summary

Today you learned how to approach common data-quality problems using a simple workflow:

**Inspect → Decide → Clean → Verify**

You practiced:

- cleaning column names;
- standardizing categorical values;
- manipulating strings;
- converting data types;
- handling missing values;
- removing duplicates;
- dropping unnecessary columns;
- comparing `map()` and `apply()`;
- exporting data with `to_csv()`.

### Further exploration

Pandas provides additional options for advanced cleaning, including:

- `dropna()` with `how` and `thresh`;
- forward fill and backward fill;
- more advanced `apply()` transformations;
- regular expressions;
- date handling;
- advanced imputation strategies.

These tools can be introduced when a dataset requires them.

---

## Additional material — Reusing your own Python functions

As your projects grow, you may write functions that are useful in several notebooks. Instead of copying the same code repeatedly, you can save those functions in a **Python module**: an ordinary `.py` file.

For example, imagine this folder structure:

```text
my_project/
├── analysis.ipynb
└── functions.py
```

Both files are in the same directory. In `functions.py`, define reusable functions using normal Python syntax. **Do not** include notebook-specific commands such as `display()` unless they are intentionally part of the function's behavior.

### Step 1 — Create `functions.py`

Using your code editor, create a file named `functions.py` in the same directory as your notebook. Add the following code:

```python
# functions.py

def normalize_city(city):
    if not isinstance(city, str):
        return city

    return city.strip().title()


def euros_to_cents(amount):
    return round(amount * 100)
```

Save the file. You now have a small module containing two functions.

**Note:** The following notebook examples assume that you have actually created and saved `functions.py`. They are not meant to be run before that step.

### Step 2 — Import one function

Use `from functions import normalize_city` when you want to bring **one specific function** into your notebook's namespace:

```python
from functions import normalize_city

normalize_city("  bArCeLoNa ")
```

You call the function directly by its name: `normalize_city(...)`.

### Step 3 — Import the entire module

Alternatively, use `import functions`:

```python
import functions

functions.normalize_city("  bArCeLoNa ")
functions.euros_to_cents(12.50)
```

Here, the functions remain under the **module name**. You must write `functions.normalize_city(...)` rather than `normalize_city(...)` (unless you also imported the function separately).

| Import statement | How to call the function |
|---|---|
| `from functions import normalize_city` | `normalize_city("madrid")` |
| `import functions` | `functions.normalize_city("madrid")` |

Neither approach is universally better. Importing the module makes it particularly clear where the function comes from; importing a specific function can be convenient when you use it frequently.

### Step 4 — Apply your function to a Pandas column

Once the function is imported, you can reuse it just like any other Python function:

```python
from functions import normalize_city

example_cities = pd.Series([" madrid ", "BARCELONA", None])
example_cities.map(normalize_city)
```

This also connects with the earlier comparison between `.map()` and `.apply()`.

### Troubleshooting

- **`ModuleNotFoundError`**: check the file name (`functions.py`) and that the notebook's current working directory contains it. You can inspect that directory with `import os` and `os.getcwd()`.
- **`AttributeError` or an old function definition after editing the file**: Python caches imported modules. For beginners, the simplest reliable solution is to **restart the notebook kernel and run the imports again**.
- **Avoid naming your file `pandas.py`, `numpy.py` or `math.py`**, as that may shadow libraries you intend to import.

**Optional practice:** Move one small function you have already written into `functions.py`, import it in your notebook, and verify that it works.

### Additional material — Reading Excel files

Real-world datasets are often distributed as Excel workbooks rather than CSV files. Pandas can read a worksheet directly into a DataFrame using `pd.read_excel()`.

```python
import pandas as pd

# First worksheet (worksheet positions start at 0)
df = pd.read_excel("shark_attacks.xlsx", sheet_name=0)

# Alternatively, select a worksheet by its name
df = pd.read_excel("shark_attacks.xlsx", sheet_name="Sheet1")
```

**Important details:**

- The argument is **`sheet_name`**, not `sheet`. `sheet_name=0` means the first worksheet; a string selects a worksheet by its exact name.
- For modern **`.xlsx`** workbooks, Pandas normally uses the **`openpyxl`** engine (which must be installed).
- For older **`.xls`** workbooks, use **`xlrd`** (which must be installed):

```python
df = pd.read_excel("shark_attacks.xls", sheet_name=0, engine="xlrd")
```

You can also specify `engine="openpyxl"` explicitly for `.xlsx` files, but this is usually unnecessary. Once loaded, the result is an ordinary DataFrame, so the same inspection and cleaning methods apply.

*The filenames and worksheet name above are illustrative: use the actual file extension and worksheet name in your project.*